# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Loaded:", len(df), "rows")

Cloning into 'flyrank_internship'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 141 (delta 48), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.84 MiB | 8.32 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/flyrank_internship
Loaded: 30000 rows


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
low_ctr = (df['ctr'] < df['ctr'].median()).astype(int)

print("Stale pages:", stale.sum())
print("Visible pages:", visible.sum())
print("Low-CTR pages:", low_ctr.sum())


Stale pages: 174
Visible pages: 16726
Low-CTR pages: 14810


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
low_ctr = (df['ctr'] < df['ctr'].median()).astype(int)

df['baseline_score'] = stale * visible * (1 + low_ctr) * df['impressions_90d']

def reason_code(row):
    codes = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        codes.append('stale_but_visible')
    if row['ctr'] < df['ctr'].median() and row['impressions_90d'] >= 500:
        codes.append('low_ctr_visible_page')
    return ','.join(codes) if codes else 'no_flag'

df['reason_code'] = df.apply(reason_code, axis=1)

queue = df[['content_id','client_id','baseline_score','reason_code',
            'impressions_90d','ctr','days_since_last_update','is_declining_label']]
queue = queue.sort_values('baseline_score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue.head(10)


Saved 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_score,reason_code,impressions_90d,ctr,days_since_last_update,is_declining_label
0,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_but_visible,61678,0.15,194,1
1,content_7368877ea310,client_7f2253d7e2,59472,stale_but_visible,59472,0.13,194,1
2,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_but_visible,25715,0.23,194,1
3,content_5feee3994adb,client_7f2253d7e2,15624,"stale_but_visible,low_ctr_visible_page",7812,0.01,194,1
4,content_0a91db491d14,client_7f2253d7e2,13299,stale_but_visible,13299,0.49,193,1
5,content_b16bd7307b39,client_7f2253d7e2,9180,"stale_but_visible,low_ctr_visible_page",4590,0.00,194,1
6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_but_visible,7558,0.20,193,1
7,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_but_visible,4556,0.33,194,1
8,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_but_visible,4429,0.38,194,1
9,content_928af3e22c80,client_7f2253d7e2,1697,stale_but_visible,1697,0.12,193,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['is_declining_label'].mean()
p_at_20 = precision_at_k(df['baseline_score'], df['is_declining_label'], 20)
p_at_50 = precision_at_k(df['baseline_score'], df['is_declining_label'], 50)

print("Base rate (declining %):", base_rate)
print("Precision@20:", p_at_20)
print("Precision@50:", p_at_50)

top20 = queue.head(20)
print(top20[['content_id','baseline_score','reason_code','impressions_90d','ctr','is_declining_label']])


Base rate (declining %): 0.5420666666666667
Precision@20: 0.9
Precision@50: 0.68
              content_id  baseline_score  \
0   content_cf56e2e2e282           61678   
1   content_7368877ea310           59472   
2   content_1bfaa38ff26c           25715   
3   content_5feee3994adb           15624   
4   content_0a91db491d14           13299   
5   content_b16bd7307b39            9180   
6   content_c2d929d83eaa            7558   
7   content_fe16a55cd13d            4556   
8   content_ecb6215e79fd            4429   
9   content_928af3e22c80            1697   
10  content_e3ff1b093148            1408   
11  content_bdbec75c1148            1316   
12  content_074ba6ead17b            1066   
13  content_7f116ae1f6f5             954   
14  content_77d4d5930e5e             828   
15  content_72496874f806             821   
16  content_6226ee6adc91             545   
17  content_e3393b0b5359               0   
18  content_f21e4a700f92               0   
19  content_ccaae106ecb6               

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
weak_picks = top20[top20['is_declining_label'] == 0]
print("Weak picks in top 20 (flagged high, but not actually declining):")
print(weak_picks[['content_id','baseline_score','reason_code','ctr','impressions_90d']])

score_inputs = ['days_since_last_update', 'impressions_90d', 'ctr']
leaky = [c for c in score_inputs if c in ['trend_direction','trend_pct','provider_used','model_used',
    'impressions_last_30d','clicks_last_30d','sessions_last_30d']]
print("\nAny leaky/product-flag columns used in the score?:", leaky)


Weak picks in top 20 (flagged high, but not actually declining):
              content_id  baseline_score           reason_code   ctr  \
11  content_bdbec75c1148            1316     stale_but_visible  0.15   
17  content_e3393b0b5359               0               no_flag  0.00   
19  content_ccaae106ecb6               0  low_ctr_visible_page  0.06   

    impressions_90d  
11             1316  
17              457  
19            43654  

Any leaky/product-flag columns used in the score?: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.